In [1]:
using Pkg
Pkg.instantiate()
Pkg.update()

    Updating registry at `~/.julia/registries/General.toml`
    Updating git-repo `https://github.com/euriqa-brassboard/MSSim.jl.git`
     Project No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Project.toml`
    Manifest No packages added to or removed from `~/projects/yyc-data/euriqa/calculations/rydberg_czs/Manifest.toml`
        Info We haven't cleaned this depot up for a bit, running Pkg.gc()...
      Active manifest files: 9 found
      Active artifact files: 1 found
      Active scratchspaces: 0 found
     Deleted no artifacts, repos, packages or scratchspaces


In [2]:
include("sqrt_cz.jl")

opt_n! (generic function with 1 method)

In [3]:
using NPZ

In [4]:
Ω = 2π * 3
nseg = 30
nsubsample = 30
t_gate = 1
t_ramp = 0.01
fm_limit = 2π * 10 * 1.5
opt = SplineOpt(Ω=Ω, nseg=nseg, nsubsample=nsubsample, t_gate=t_gate, t_ramp=t_ramp,
                lam_rob=0.0, lam_leak=0.2, lam_dark=0.0, fm_limit=fm_limit);
# opt = Opt(Ω=Ω, num_slices=num_slices, t_gate=t_gate,
#           lam_rob=0.1, lam_leak=1, lam_dark=1);
# optional keyword arguments:
# algorithm=:LD_CCSAQ, maxeval_pre=1000, maxtime=3, xtol=1e-7, minω=-2π * 10, maxω=2π * 10

In [5]:
best_obj, best_args = @time opt_n!(opt, 40; verbose=false, pre_threshold=0.02) # default verbosity is true

obj = 0.17147247852012562
obj = 0.012786502367938402
Round 20 done
obj = 0.0011907150696108656
Round 40 done
 34.378727 seconds (2.22 M allocations: 67.472 MiB, 0.05% gc time, 0.76% compilation time)


(0.0011907150696108656, [-16.7463413962334, -13.604748742648367, -10.463156089067692, -7.321563435513888, -4.179970781957663, -1.0383781283885822, -1.1919597235316761, -4.333552377121009, -5.317619817887824, -2.1760271642980324  …  -5.5446656485892, -2.403073123910447, 0.7385195888009454, -0.2055745244533937, -3.3471671780105003, -6.488759831434136, -9.630352484939221, -12.771945135040268, -15.913537788450055, -2.338953961772381])

In [6]:
fm_rate_constraints(best_args, (), 1:opt.idx_max, opt.diff_limit)

1.8744357372924014e-6

In [ ]:
# More tries to refine the result.
for _ in 1:10
    best_obj, best_args = @time opt_n!(opt, 40; pre_threshold=0.02,
                                       verbose=false, best_obj=best_obj, best_args=best_args)
    if best_obj < 1e-4
        break
    end
end

Round 20 done
Round 40 done
 17.171538 seconds (1.88 M allocations: 47.945 MiB, 0.05% gc time, 0.05% compilation time)
obj = 0.0011906732325567054
Round 20 done
Round 40 done
 33.786685 seconds (1.92 M allocations: 48.753 MiB, 0.02% gc time)
Round 20 done
Round 40 done
 30.179585 seconds (3.15 M allocations: 80.066 MiB, 0.03% gc time)
Round 20 done
Round 40 done
 30.249815 seconds (2.10 M allocations: 53.328 MiB, 0.03% gc time)
Round 20 done
Round 40 done
 26.312421 seconds (2.29 M allocations: 58.155 MiB, 0.02% gc time)
Round 20 done
Round 40 done
 22.607282 seconds (2.09 M allocations: 53.036 MiB, 0.02% gc time)
Round 20 done
Round 40 done
 31.887652 seconds (3.12 M allocations: 79.238 MiB, 0.03% gc time)
Round 20 done
Round 40 done
 23.824077 seconds (1.26 M allocations: 32.142 MiB, 0.02% gc time)
Round 20 done
Round 40 done
 22.521308 seconds (1.73 M allocations: 44.122 MiB, 0.02% gc time)
Round 20 done


In [ ]:
best_ϕs = fm_to_phase(opt, best_args)
println(best_ϕs)

In [ ]:
npzwrite("sqrtcz_0.6us_3MHz.npz", Dict("phase_list"=>best_ϕs, "t_gate"=>t_gate, "t_ramp"=>t_ramp, "Omega"=>Ω, "Omegas"=>Vector(opt.cb.Ωs)))